In [17]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.preprocessing import Normalizer
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('plan_purchase.csv')
print(df.head(5))
X = df.drop("Purchase", axis=1)
y = df["Purchase"].map({"No": 0, "Yes": 1})  # Convert 'Yes'/'No' to 1/0

X
y

   Age  MonthlyIncome  PlanType  UsageScore Purchase
0   56          81476  Standard          90      Yes
1   46          64811  Standard          92      Yes
2   32          56208     Basic          71      Yes
3   25          40150   Premium          82      Yes
4   38          63286  Standard          34       No


0      1
1      1
2      1
3      1
4      0
      ..
495    0
496    0
497    1
498    1
499    0
Name: Purchase, Length: 500, dtype: int64

In [3]:
categorical_features = X.select_dtypes(include='object').columns
numeric_features = X.select_dtypes(exclude='object').columns
print("Categorical Features:", list(categorical_features))
print("Numerical Features:", list(numeric_features))

# numeric pipeline: clean -> scale -> normalize
numerical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median'))
])

Categorical Features: ['PlanType']
Numerical Features: ['Age', 'MonthlyIncome', 'UsageScore']


In [4]:
# numeric pipeline: clean -> scale -> normalize
numerical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median'))
])
# For random forest Standard scaler and normalizer is not needed

# categorical pipeline: clean -> encode
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])

# combine both, variable, pipeline,feature_list
preprocessor = ColumnTransformer([
    ('num', numerical_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

In [7]:
# full pipeline: preprocess + model
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Inspect columns and basic info
print("Columns:", df.columns.tolist())
print("\nInfo:")
print(df.info())
print("\nDescribe:")
display(df.describe())

Columns: ['Age', 'MonthlyIncome', 'PlanType', 'UsageScore', 'Purchase']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Age            500 non-null    int64 
 1   MonthlyIncome  500 non-null    int64 
 2   PlanType       500 non-null    object
 3   UsageScore     500 non-null    int64 
 4   Purchase       500 non-null    object
dtypes: int64(3), object(2)
memory usage: 19.7+ KB
None

Describe:


,Age,MonthlyIncome,UsageScore
count,500.000000,500.000000,500.000000
mean,39.326000,52753.620000,60.082000
std,12.200386,20181.171598,19.938967
min,18.000000,20055.000000,0.000000
25%,29.000000,35309.500000,46.000000
50%,41.000000,52286.000000,61.000000
75%,50.000000,70364.250000,75.000000
max,59.000000,89896.000000,100.000000


In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y)

print("Train class distributiion:")
print(y_train.value_counts(normalize=True))

print("\nTest class distributiion:")
print(y_test.value_counts(normalize=True))


Train class distributiion:
Purchase
0    0.562857
1    0.437143
Name: proportion, dtype: float64

Test class distributiion:
Purchase
0    0.566667
1    0.433333
Name: proportion, dtype: float64


In [19]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['Age', 'MonthlyIncome', 'UsageScore'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['PlanType'], dtype='object'))])),
                ('model', RandomForestClassifier(random_state=42))])

In [22]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test,y_pred))
print("Report:", classification_report(y_test,y_pred))

Accuracy: 0.9866666666666667
Report:               precision    recall  f1-score   support

           0       0.98      1.00      0.99        85
           1       1.00      0.97      0.98        65

    accuracy                           0.99       150
   macro avg       0.99      0.98      0.99       150
weighted avg       0.99      0.99      0.99       150



In [23]:
import joblib as jb

jb.dump(pipeline,"Pipeline.pkl")

['Pipeline.pkl']